In [2]:
import pandas as pd

# 1. 저장된 파일 불러오기
df_final = pd.read_csv("city_temperature_cleaned.csv", parse_dates=['date'])

print("--- [최종 데이터 상태 보고서] ---")

# 검증 1: 데이터 크기 및 결측치
print(f"1. 전체 데이터 수: {len(df_final)}개")
print(f"2. 전체 결측치 수:\n{df_final.isna().sum()}")

# 검증 2: 연도 범위 (200, 201이 사라졌는지 확인)
print(f"\n3. 연도 범위: {df_final['Year'].min()} ~ {df_final['Year'].max()}")

# 검증 3: 온도 범위 (이상치 -99.0 및 -50 이하가 사라졌는지 확인)
print(f"4. 온도 범위 (°F): {df_final['AvgTemperature'].min()} ~ {df_final['AvgTemperature'].max()}")

# 검증 4: 중복 데이터 (0건이어야 함)
dup_count = df_final.duplicated(subset=['City', 'date']).sum()
print(f"5. 도시별 중복 날짜 데이터 수: {dup_count}건")

# 검증 5: 데이터 타입 확인 (date가 datetime64인지 확인)
print(f"6. 데이터 타입:\n{df_final.dtypes}")

C:\Users\user\AppData\Local\Temp\ipykernel_1712\3719975390.py:4: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df_final = pd.read_csv("city_temperature_cleaned.csv", parse_dates=['date'])


--- [최종 데이터 상태 보고서] ---
1. 전체 데이터 수: 2769406개
2. 전체 결측치 수:
Region                  0
Country                 0
State             1374265
City                    0
Month                   0
Day                     0
Year                    0
AvgTemperature          0
date                    0
dtype: int64

3. 연도 범위: 1995 ~ 2020
4. 온도 범위 (°F): -49.1 ~ 110.0
5. 도시별 중복 날짜 데이터 수: 0건
6. 데이터 타입:
Region                    object
Country                   object
State                     object
City                      object
Month                      int64
Day                        int64
Year                       int64
AvgTemperature           float64
date              datetime64[ns]
dtype: object


In [3]:
df_final["City"].nunique()

321

In [4]:
city_counts = df_final.groupby("City").size()
print(city_counts.describe())

count     321.000000
mean     8627.433022
std      1516.426402
min       884.000000
25%      9119.000000
50%      9234.000000
75%      9246.000000
max      9254.000000
dtype: float64


In [5]:
# 데이터 개수가 적은(예: 9000개 미만) 도시들 확인
low_data_cities = city_counts[city_counts < 9000]
print("--- 데이터가 부족한 도시 목록 ---")
print(low_data_cities.sort_values())

--- 데이터가 부족한 도시 목록 ---
City
Bujumbura       884
Bonn           1083
Georgetown     2136
Frankfurt      2331
Lilongwe       2641
               ... 
Belize City    8867
Sofia          8871
Colombo        8896
Islamabad      8932
Cotonou        8981
Length: 69, dtype: int64


In [6]:
# 데이터 개수가 적은(예: 8000개 미만) 도시들 확인
low_data_cities = city_counts[city_counts < 8000]
print("--- 데이터가 부족한 도시 목록 ---")
print(low_data_cities.sort_values())

--- 데이터가 부족한 도시 목록 ---
City
Bujumbura                884
Bonn                    1083
Georgetown              2136
Frankfurt               2331
Lilongwe                2641
Freetown                3064
Yerevan                 3188
Pristina                3247
Hamburg                 3445
Flagstaff               3545
Dhaka                   3567
Yuma                    4087
Tbilisi                 4101
Nicosia                 4156
Tel Aviv                4196
Bissau                  5181
Guadalajara             5181
Hamilton                5558
Daytona Beach           5699
Montreal                5699
Wilmington              5724
Harrisburg              5736
Baton Rouge             5752
Bangkok                 5763
Port au Prince          5816
Muscat                  5924
Nairobi                 6031
Lagos                   6036
San Juan Puerto Rico    6088
Athens                  6350
Lusaka                  6378
Quito                   6594
Jakarta                 6858
Calgary        

In [7]:
# 1. 8000개 이상 데이터를 가진 도시 리스트업
valid_cities = city_counts[city_counts >= 8000].index

# 2. 해당 도시들만 필터링한 최종 분석용 데이터프레임
df_analysis = df_final[df_final['City'].isin(valid_cities)].copy()

print(f"최종 분석 대상 도시 수: {df_analysis['City'].nunique()}개")

최종 분석 대상 도시 수: 273개


In [8]:
city = df_final[df_final["City"]=="Tokyo"]

city = city.sort_values("date")

missing = city["date"].diff().value_counts()

print(missing.head())

date
1 days    9216
2 days      16
3 days       2
5 days       2
Name: count, dtype: int64


In [9]:
gap_stats = []

for city_name, group in df_final.groupby("City"):
    gaps = group.sort_values("date")["date"].diff().dt.days
    max_gap = gaps.max()
    gap_stats.append((city_name, max_gap))

gap_df = pd.DataFrame(gap_stats, columns=["City", "MaxGapDays"])
print(gap_df.sort_values("MaxGapDays", ascending=False).head(10))

               City  MaxGapDays
40           Bissau      3003.0
117        Freetown      2040.0
201          Muscat      1924.0
59        Bujumbura      1595.0
166        Lilongwe      1412.0
120      Georgetown      1318.0
202         Nairobi      1105.0
26           Banjul       759.0
229  Port au Prince       753.0
127     Guadalajara       741.0


In [10]:
df_final["City"].nunique()

321

In [11]:
df_final["City"].value_counts().head()

City
Columbus       9254
Shreveport     9253
Charleston     9253
Little Rock    9252
San Diego      9252
Name: count, dtype: int64

In [12]:
df_final.groupby("City")["AvgTemperature"].mean().sort_values().head()

City
Fairbanks     28.462646
Ulan-bator    30.323132
Regina        37.268836
Winnipeg      37.814447
Anchorage     37.942650
Name: AvgTemperature, dtype: float64

In [13]:
df_final.groupby("City")["AvgTemperature"].std().sort_values().head()

City
Bogota          1.840934
Bridgetown      1.885918
Panama City     2.074660
Kuala Lumpur    2.152811
San Jose        2.161664
Name: AvgTemperature, dtype: float64

In [15]:
# 1. 8000개 이상 데이터를 가진 도시 리스트업
valid_cities = city_counts[city_counts >= 8000].index

# 2. 해당 도시들만 필터링한 최종 분석용 데이터프레임
df_analysis = df_final[df_final['City'].isin(valid_cities)].copy()

print(f"최종 분석 대상 도시 수: {df_analysis['City'].nunique()}개")

최종 분석 대상 도시 수: 273개


In [14]:
# 1. 데이터가 8,000개 이상인 도시 리스트
valid_count_cities = city_counts[city_counts >= 8000].index

# 2. 날짜 간격(Gap)이 1년(365일) 미만인 도시 리스트 (방금 만드신 gap_df 활용)
valid_gap_cities = gap_df[gap_df["MaxGapDays"] < 365]["City"]

# 3. 두 조건을 모두 만족하는 '우량 도시'만 추출
final_valid_cities = list(set(valid_count_cities) & set(valid_gap_cities))

df_final_clean = df_final[df_final["City"].isin(final_valid_cities)].copy()

print(f"최종 분석 대상 도시 수: {len(final_valid_cities)}개")

최종 분석 대상 도시 수: 272개


In [17]:
# 8000개 이상이지만 Gap이 365일 이상인 그 도시 찾기
diff_city = set(valid_count_cities) - set(final_valid_cities)
print(f"필터링에서 탈락한 1개의 도시는: {diff_city}")

# 그 도시의 MaxGapDays가 얼마였는지 확인
print(gap_df[gap_df["City"].isin(diff_city)])

필터링에서 탈락한 1개의 도시는: {'Sofia'}
      City  MaxGapDays
276  Sofia       367.0


In [18]:
# 1. 272개 정예 도시 데이터만 추출 (df_final_clean 생성)
df_final_clean = df_final[df_final["City"].isin(final_valid_cities)].copy()

# 2. 파일 저장 (이름을 구분하기 쉽게 'final_272'를 붙였습니다)
df_final_clean.to_csv("city_temperature_final_272.csv", index=False)

print("--- [최종 저장 완료] ---")
print(f"대상 도시 수: {df_final_clean['City'].nunique()}개")
print(f"전체 행 수: {len(df_final_clean)}행")
print("파일명: city_temperature_final_272.csv")

--- [최종 저장 완료] ---
대상 도시 수: 272개
전체 행 수: 2497894행
파일명: city_temperature_final_272.csv
